# Day 33: INT8 Quantization Pipeline — Quantize, Dequantize, Measure Error

**Layer:** Implementation | **Prerequisite:** Day 11 (Quantization Formats)


## Concept Overview

An INT8 quantization pipeline has three stages: calibration (find the optimal scale/zero-point from representative data), quantization (convert FP16 weights/activations to INT8), and dequantization (scale INT8 back to FP16 for computation). The challenge: minimizing quantization error while keeping the pipeline fast.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Symmetric and Asymmetric Quantization


In [ ]:
def quantize_symmetric(x, n_bits=8):
    qmax = 2**(n_bits-1) - 1
    scale = x.abs().max() / qmax
    x_q = torch.clamp(torch.round(x / scale), -qmax, qmax).to(torch.int8)
    return x_q, scale

def dequantize_symmetric(x_q, scale):
    return x_q.float() * scale

def quantize_asymmetric(x, n_bits=8):
    qmin, qmax = 0, 2**n_bits - 1
    x_min, x_max = x.min().item(), x.max().item()
    scale = (x_max - x_min) / (qmax - qmin)
    zero_point = round(-x_min / scale)
    x_q = torch.clamp(torch.round(x / scale + zero_point), qmin, qmax).to(torch.uint8)
    return x_q, scale, zero_point

def dequantize_asymmetric(x_q, scale, zero_point):
    return (x_q.float() - zero_point) * scale

# Compare on realistic weight and activation distributions
torch.manual_seed(42)
weights = torch.randn(512, 512) * 0.02       # ~Gaussian, symmetric
activations = torch.randn(64, 512).abs() * 0.5  # ReLU output: non-negative

for name, tensor, is_symmetric in [('Weights (symmetric)', weights, True),
                                    ('Activations (asymmetric)', activations, False)]:
    if is_symmetric:
        q, s = quantize_symmetric(tensor)
        dq = dequantize_symmetric(q, s)
    else:
        q, s, zp = quantize_asymmetric(tensor)
        dq = dequantize_asymmetric(q, s, zp)
    err = (tensor - dq).abs()
    print(f'{name}:')
    print(f'  Scale: {s:.6f}')
    print(f'  MAE: {err.mean():.6f} ({err.mean()/tensor.abs().mean()*100:.2f}% of mean magnitude)')
    print(f'  Max error: {err.max():.6f}')
    print()


## 2. Per-Channel vs Per-Tensor Quantization


In [ ]:
def quantize_per_channel(W, n_bits=8):
    """Quantize each output channel with its own scale."""
    qmax = 2**(n_bits-1) - 1
    scale = W.abs().max(dim=1, keepdim=True).values / qmax
    W_q = torch.clamp(torch.round(W / scale), -qmax, qmax).to(torch.int8)
    return W_q, scale

# Test on a weight matrix with varying channel magnitudes
np.random.seed(42)
W = torch.randn(256, 512) * 0.01
W[::8] *= 20  # every 8th row has large values (outlier channels)

W_q_tensor, s_tensor = quantize_symmetric(W)
W_dq_tensor = dequantize_symmetric(W_q_tensor, s_tensor)

W_q_channel, s_channel = quantize_per_channel(W)
W_dq_channel = (W_q_channel.float() * s_channel)

err_tensor = (W - W_dq_tensor).abs().mean().item()
err_channel = (W - W_dq_channel).abs().mean().item()

print(f'Per-tensor INT8 MAE:  {err_tensor:.6f}')
print(f'Per-channel INT8 MAE: {err_channel:.6f}')
print(f'Improvement: {err_tensor/err_channel:.1f}x')

# Visualize scale distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(W.abs().max(dim=1).values.numpy(), bins=40)
ax1.set_title('Per-channel weight magnitude'); ax1.set_xlabel('Max |weight|')
ax2.bar(range(0, 256, 8), s_channel[::8, 0].detach().numpy(), alpha=0.7, label='Outlier channels')
ax2.bar([i for i in range(256) if i%8!=0],
        s_channel[[i for i in range(256) if i%8!=0], 0].detach().numpy(), alpha=0.7, label='Normal channels')
ax2.set_title('Per-channel scales'); ax2.set_xlabel('Channel index'); ax2.legend()
plt.tight_layout()
plt.savefig('per_channel_quant.png', dpi=100, bbox_inches='tight')
plt.show()


## 3. Quantized Linear Layer


In [ ]:
class QuantizedLinear(nn.Module):
    def __init__(self, fp16_weight, n_bits=8):
        super().__init__()
        W_q, scale = quantize_per_channel(fp16_weight, n_bits)
        self.register_buffer('W_q', W_q)
        self.register_buffer('scale', scale)
        self.n_bits = n_bits

    def forward(self, x):
        # Dequantize weights and compute matmul in FP16
        W_fp = self.W_q.float() * self.scale
        return F.linear(x, W_fp)

# Compare FP16 vs INT8 linear layer
fp16_linear = nn.Linear(512, 256, bias=False)
int8_linear = QuantizedLinear(fp16_linear.weight)
x = torch.randn(32, 512)

out_fp16 = fp16_linear(x)
out_int8 = int8_linear(x.float()).half()
err = (out_fp16 - out_int8.float()).abs()

print(f'FP16 vs INT8 linear layer output error:')
print(f'  MAE: {err.mean():.6f}')
print(f'  Relative error: {err.mean()/out_fp16.abs().mean()*100:.2f}%')

import os
fp16_size = sum(p.numel() * 2 for p in fp16_linear.parameters())
int8_size = sum(p.numel() * 1 for p in int8_linear.buffers() if p.dtype == torch.int8)
print(f'\nMemory: FP16={fp16_size/1024:.0f}KB INT8≈{int8_size/1024:.0f}KB ({fp16_size/int8_size:.1f}x reduction)')


## Experiments

1. Implement calibration: collect activation statistics from a forward pass on a calibration dataset and use them to set quantization ranges.
2. Measure actual throughput of INT8 matmul vs FP16 on your GPU.
3. Quantize all layers of the MinimalDecoder from Day 29 and measure perplexity change.


## Key Takeaways

- Symmetric quantization is simpler but suboptimal for non-negative tensors (activations); asymmetric uses the full range.
- Per-channel quantization reduces error by 10-100x vs per-tensor for weights with outlier channels.
- The INT8 pipeline: calibrate scales → quantize weights offline → at inference, dequantize to FP16 for matmul (weight-only) or use INT8 GEMM (W8A8).
- Quantization error of ~1% relative is generally acceptable for most downstream tasks; >5% typically causes noticeable quality degradation.

## References
- Dettmers et al. (2022), "LLM.int8()"
- Jacob et al. (2018), "Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference"
